In [0]:
%sql
-- List all catalogs available in my workspace
SHOW CATALOGS

In [0]:
%sql
-- Create one schema (drawer) for each layer of the medallion architecture
-- IF NOT EXISTS makes the command safe to re-run: it will not fail if the schema already exists
CREATE SCHEMA IF NOT EXISTS workspace.bronze COMMENT 'Raw data, exactly as received from the source';
CREATE SCHEMA IF NOT EXISTS workspace.silver COMMENT 'Cleaned and standardized data';
CREATE SCHEMA IF NOT EXISTS workspace.gold   COMMENT 'Aggregated data, ready for dashboards and analysis';

In [0]:
%sql
-- List all schemas inside my catalog to confirm the three layers were created
SHOW SCHEMAS IN workspace

In [0]:
%sql
-- Create a volume (file folder) inside the bronze schema to receive raw files
CREATE VOLUME IF NOT EXISTS workspace.bronze.landing
COMMENT 'Landing zone: raw files exactly as delivered by the source system'

In [0]:

#Criacao de dados fake para teste

import csv
import random
from datetime import datetime, timedelta

# Fixed seed: every run generates the same data (makes the project reproducible)
random.seed(42)

# Path of the file inside the volume (format: /Volumes/catalog/schema/volume/file)
file_path = "/Volumes/workspace/bronze/landing/transactions_2026_10_10.csv"

categories = ["groceries", "transport", "salary", "rent", "leisure", "health"]
statuses = ["completed", "pending", "failed"]
start_date = datetime(2026, 9, 1)

rows = []
for i in range(1, 501):
    # Random date within ~40 days after the start date
    date = start_date + timedelta(days=random.randint(0, 40))
    amount = round(random.uniform(5, 3000), 2)

    row = {
        "transaction_id": f"TX{i:05d}",
        "account_id": f"ACC{random.randint(1, 50):03d}",
        "transaction_date": date.strftime("%Y-%m-%d"),
        "amount": amount,
        "currency": "BRL",
        "category": random.choice(categories),
        "status": random.choice(statuses),
    }
    rows.append(row)

# --- Intentional "dirty data" (to be cleaned in the silver layer) ---
rows[10]["category"] = ""                    # missing category
rows[20]["category"] = ""                    # missing category
rows[30]["status"] = rows[30]["status"].upper()   # inconsistent casing
rows[40]["amount"] = str(rows[40]["amount"]).replace(".", ",")  # comma as decimal separator
rows.extend(rows[:10])                       # 10 duplicated rows

# Write the CSV file into the volume
with open(file_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()      # first line = column names
    writer.writerows(rows)    # all other lines = data

print(f"File created with {len(rows)} rows (including duplicates)")

In [0]:
# List files in the landing zone to confirm the file arrived
display(dbutils.fs.ls("/Volumes/workspace/bronze/landing/"))